# Web Scrapping: Selenium

## 1. Libraries

In [1]:
import pandas as pd
import time

# Herramientas de Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException


## 2.  Configuración e Inicialización del Navegador

In [2]:
# Configuramos las opciones de Chrome
chrome_options = Options()
#chrome_options.add_argument("--headless")  # Descomenta esta línea si no quieres ver la ventana del navegador
chrome_options.add_argument("--start-maximized") # Inicia el navegador maximizado
chrome_options.add_argument("--lang=en-US") # Solicitamos la página en inglés para tener selectores más consistentes

# Iniciamos el WebDriver
try:
    driver = webdriver.Chrome(options=chrome_options)
    print("WebDriver iniciado con éxito.")
except Exception as e:
    print(f"Error al iniciar el WebDriver: {e}")

WebDriver iniciado con éxito.


## 3. Navegar a la Página de IMdb

In [3]:
# URL del Top 250 de IMDb
url = "https://www.imdb.com/chart/top/"
driver.get(url)

# Lista para guardar los datos de cada película
movies_data = []

# Selector CSS para la lista que contiene todas las películas
# Las 'ul' (unordered lists) suelen contener este tipo de elementos.
movie_list_selector = "ul.ipc-metadata-list"

try:
    print("Esperando a que la lista de películas cargue...")
    # Esperamos un máximo de 10 segundos a que el elemento ul sea visible
    WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, movie_list_selector))
    )
    print("Lista de películas encontrada. Comenzando el scraping.")
except TimeoutException:
    print("Error: La lista de películas no cargó a tiempo. El script se detendrá.")
    driver.quit()

Esperando a que la lista de películas cargue...
Lista de películas encontrada. Comenzando el scraping.


## 4. Bucle Principal de Scraping

In [4]:
# Selector para cada item (película) en la lista. Son elementos <li>.
movie_item_selector = "li.ipc-metadata-list-summary-item"
movie_elements = driver.find_elements(By.CSS_SELECTOR, movie_item_selector)

# Iteramos solo sobre las primeras 50 películas
for movie in movie_elements[:50]:
    try:
        # --- Rango y Título ---
        # El título está en una etiqueta <h3>. El texto es "1. The Shawshank Redemption"
        title_element = movie.find_element(By.CSS_SELECTOR, "h3.ipc-title__text")
        full_title_text = title_element.text
        # Separamos el rango del título usando el punto.
        rank, title = full_title_text.split('. ', 1)

        # --- Año, Duración y Clasificación ---
        # Estos datos están en un contenedor div con varios spans
        metadata_elements = movie.find_elements(By.CSS_SELECTOR, "div.cli-title-metadata > span")
        year = metadata_elements[0].text if len(metadata_elements) > 0 else "No disponible"
        
        # --- Calificación de IMDb ---
        # La calificación está en un span con un aria-label específico
        rating_element = movie.find_element(By.CSS_SELECTOR, "span.ipc-rating-star")
        rating = rating_element.text.split('\n')[0] # Extraemos solo la calificación numérica

        # --- URL de la película ---
        # El enlace está en la etiqueta <a> que contiene el título
        url_element = movie.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper")
        movie_url = url_element.get_attribute('href')

        # Guardamos los datos extraídos en un diccionario
        movies_data.append({
            "Rango": rank,
            "Titulo": title,
            "Año": year,
            "Calificacion_IMDb": rating,
            "URL": movie_url
        })
        print(f"Scraped: #{rank} {title}")

    except Exception as e:
        print(f" Error extrayendo datos de una película. Error: {e}")
        continue

print(f"\nScraping completado. Se extrajeron datos de {len(movies_data)} películas.")

Scraped: #1 The Shawshank Redemption
Scraped: #2 The Godfather
Scraped: #3 The Dark Knight
Scraped: #4 The Godfather Part II
Scraped: #5 12 Angry Men
Scraped: #6 The Lord of the Rings: The Return of the King
Scraped: #7 Schindler's List
Scraped: #8 Pulp Fiction
Scraped: #9 The Lord of the Rings: The Fellowship of the Ring
Scraped: #10 The Good, the Bad and the Ugly
Scraped: #11 Forrest Gump
Scraped: #12 The Lord of the Rings: The Two Towers
Scraped: #13 Fight Club
Scraped: #14 Inception
Scraped: #15 Star Wars: Episode V - The Empire Strikes Back
Scraped: #16 The Matrix
Scraped: #17 Goodfellas
Scraped: #18 Interstellar
Scraped: #19 One Flew Over the Cuckoo's Nest
Scraped: #20 Se7en
Scraped: #21 It's a Wonderful Life
Scraped: #22 The Silence of the Lambs
Scraped: #23 Seven Samurai
Scraped: #24 Saving Private Ryan
Scraped: #25 The Green Mile
Scraped: #26 City of God
Scraped: #27 Life Is Beautiful
Scraped: #28 Terminator 2: Judgment Day
Scraped: #29 Star Wars: Episode IV - A New Hope
Scrap

## 5. Crear el DataFrame y Guardar los Datos

In [ ]:
if movies_data:
    # Creamos el DataFrame
    df = pd.DataFrame(movies_data)
    
    # Guardamos el DataFrame en un archivo CSV
    # encoding='utf-8-sig' ayuda a evitar problemas con caracteres especiales
    df.to_csv("imdb_top_50_peliculas.csv", index=False, encoding='utf-8-sig')
    
    print("\n Datos guardados exitosamente en 'imdb_top_50_peliculas.csv'")
    
    # Mostramos las primeras 5 filas del DataFrame para verificar
    display(df.head())
else:
    print("\nNo se pudo extraer ningún dato de las películas.")

# Cerramos el navegador
driver.quit()
print("\nNavegador cerrado correctamente.")


🎉 Datos guardados exitosamente en 'imdb_top_50_peliculas.csv'


,Rango,Titulo,Año,Calificacion_IMDb,URL
0,1,The Shawshank Redemption,1994,9.3,https://www.imdb.com/title/tt0111161/?ref_=cht...
1,2,The Godfather,1972,9.2,https://www.imdb.com/title/tt0068646/?ref_=cht...
2,3,The Dark Knight,2008,9.1,https://www.imdb.com/title/tt0468569/?ref_=cht...
3,4,The Godfather Part II,1974,9.0,https://www.imdb.com/title/tt0071562/?ref_=cht...
4,5,12 Angry Men,1957,9.0,https://www.imdb.com/title/tt0050083/?ref_=cht...



Navegador cerrado correctamente.
